In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm import tqdm


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-data-root", type=Path, default=Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export"))
    parser.add_argument("--train-subdir", type=Path, default=Path("reduced_64x32/train"))
    parser.add_argument("--test-subdir", type=Path, default=Path("reduced_64x32/test"))
    parser.add_argument("--output", type=Path, default=Path("drive/MyDrive/kaggle_cs3780_sp26/reduced_64x32_logreg_metrics_solution_eval.json"))
    parser.add_argument("--solution-csv", type=Path, default=Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv"))
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--max-iter", type=int, default=1000)
    args, unknown = parser.parse_known_args()
    return args


def load_variant_dataset(variant_dir: Path, include_paths: bool = False) -> tuple[np.ndarray, np.ndarray | list[Path], tuple[int, int]]:
    features: list[np.ndarray] = []
    labels_or_paths: list[str | Path] = []
    image_size: tuple[int, int] | None = None

    image_paths = sorted(variant_dir.rglob("*.png"))
    if not image_paths:
        raise FileNotFoundError(f"No PNG files found under {variant_dir}")

    for image_path in tqdm(image_paths, desc=f"Loading {variant_dir.name}", unit="img"):
        with Image.open(image_path) as image:
            gray = image.convert("L")
            if image_size is None:
                image_size = gray.size
            elif gray.size != image_size:
                gray = gray.resize(image_size, Image.LANCZOS)
            features.append(np.asarray(gray, dtype=np.float32).reshape(-1))
        if include_paths:
            labels_or_paths.append(image_path)
        else:
            labels_or_paths.append(image_path.parent.name)

    return np.stack(features), np.array(labels_or_paths) if not include_paths else labels_or_paths, image_size


def train_and_evaluate_separate_datasets(
    x_train_raw: np.ndarray,
    y_train_raw: np.ndarray,
    x_test_raw: np.ndarray,
    y_test_solution_labels: np.ndarray,
    seed: int,
    max_iter: int,
) -> dict:
    print(f"[start] training {x_train_raw.shape[1]}-dim logistic regression with external test set", flush=True)

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train_raw)
    y_test_encoded = label_encoder.transform(y_test_solution_labels)

    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "logreg",
                SGDClassifier(
                    loss='log_loss',
                    max_iter=max_iter,
                    random_state=seed,
                    n_jobs=1,
                    verbose=1,
                    early_stopping=True,
                    validation_fraction=0.1,
                    n_iter_no_change=5,
                ),
            ),
        ]
    )
    model.fit(x_train_raw, y_train_encoded)
    print(f"[done] training {x_train_raw.shape[1]}-dim logistic regression", flush=True)

    train_pred = model.predict(x_train_raw)
    test_pred = model.predict(x_test_raw)

    report = classification_report(
        y_test_encoded,
        test_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )

    return {
        "train_accuracy": float(accuracy_score(y_train_encoded, train_pred)),
        "test_accuracy": float(accuracy_score(y_test_encoded, test_pred)),
        "n_train": int(x_train_raw.shape[0]),
        "n_test": int(x_test_raw.shape[0]),
        "n_features": int(x_train_raw.shape[1]),
        "n_classes": int(len(label_encoder.classes_)),
        "classification_report": report,
    }


def main() -> None:
    args = parse_args()

    train_data_path = args.base_data_root / args.train_subdir
    test_data_path = args.base_data_root / args.test_subdir
    solution_csv_path = args.solution_csv

    if not train_data_path.exists():
        raise FileNotFoundError(f"Training data root not found: {train_data_path}")
    if not test_data_path.exists():
        raise FileNotFoundError(f"Testing data root not found: {test_data_path}")
    if not solution_csv_path.exists():
        raise FileNotFoundError(f"Solution CSV not found: {solution_csv_path}")


    print(f"Loading training data from: {train_data_path}", flush=True)
    x_train, y_train_labels, image_size_train = load_variant_dataset(train_data_path)
    print(f"Loading testing data from: {test_data_path}", flush=True)
    x_test, test_image_paths, image_size_test = load_variant_dataset(test_data_path, include_paths=True)

    if image_size_train != image_size_test:
        print(f"Warning: Training image size {image_size_train} differs from testing image size {image_size_test}. Resizing might have occurred.", flush=True)

    print(f"Loading solution data from: {solution_csv_path}", flush=True)
    solution_df = pd.read_csv(solution_csv_path)

    solution_map = solution_df.set_index('file_name')['label'].to_dict()

    y_test_solution_labels = np.array([solution_map[p.name] for p in test_image_paths])

    if len(x_test) != len(y_test_solution_labels):
        raise ValueError(f"Mismatch between number of test images ({len(x_test)}) and solution labels ({len(y_test_solution_labels)})")


    metrics = train_and_evaluate_separate_datasets(
        x_train_raw=x_train,
        y_train_raw=y_train_labels,
        x_test_raw=x_test,
        y_test_solution_labels=y_test_solution_labels,
        seed=args.seed,
        max_iter=args.max_iter,
    )

    experiment_name = f"{args.train_subdir.parent.name}"
    results = {
        "config": {
            "base_data_root": str(args.base_data_root),
            "train_subdir": str(args.train_subdir),
            "test_subdir": str(args.test_subdir),
            "solution_csv": str(args.solution_csv),
            "seed": args.seed,
            "max_iter": args.max_iter,
        },
        "experiments": {
            experiment_name: metrics
        },
    }
    results["experiments"][experiment_name]["image_size_train"] = list(image_size_train)
    results["experiments"][experiment_name]["image_size_test"] = list(image_size_test)


    print(
        f"{experiment_name}: "
        f"train={metrics['train_accuracy']:.4f}, "
        f"test={metrics['test_accuracy']:.4f}, "
        f"d={metrics['n_features']}"
    )

    with args.output.open("w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Saved metrics to {args.output}")


if __name__ == "__main__":
    main()

Loading training data from: drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/train


Loading train: 100%|██████████| 27908/27908 [1:56:06<00:00,  4.01img/s]


Loading testing data from: drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/test


Loading test: 100%|██████████| 6960/6960 [02:29<00:00, 46.48img/s] 

Loading solution data from: drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv


[start] training 2048-dim logistic regression with external test set
-- Epoch 1
Norm: 478.91, NNZs: 2048, Bias: -2413.124131, T: 25117, Avg. loss: 253.418885
Total training time: 0.10 seconds.
-- Epoch 2
Norm: 467.16, NNZs: 2048, Bias: -2107.310019, T: 50234, Avg. loss: 98.475745
Total training time: 0.22 seconds.
-- Epoch 3
Norm: 449.58, NNZs: 2048, Bias: -1927.460370, T: 75351, Avg. loss: 82.017197
Total training time: 0.34 seconds.
-- Epoch 4
Norm: 440.95, NNZs: 2048, Bias: -1798.722918, T: 100468, Avg. loss: 72.990923
Total training time: 0.51 seconds.
-- Epoch 5
Norm: 426.68, NNZs: 2048, Bias: -1700.263884, T: 125585, Avg. loss: 66.918599
Total training time: 0.63 seconds.
-- Epoch 6
Norm: 425.15, NNZs: 2048, Bias: -1617.257104, T: 150702, Avg. loss: 62.580074
Total training time: 0.75 seconds.
Convergence after 6 epochs took 0.76 seconds
-- Epoch 1
Norm: 414.63, NNZs: 2048, Bias: -2136.780740, T: 25117, Avg. loss: 300.903047
Total training time: 0.12 seconds.
-- Epoch 2
Norm: 400

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:   19.7s finished


reduced_64x32: train=0.2607, test=0.2152, d=2048
Saved metrics to drive/MyDrive/kaggle_cs3780_sp26/reduced_64x32_logreg_metrics_solution_eval.json
